In [1]:
# ============================================================
# 1. Imports
# ============================================================

import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

import glob
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

# ============================================================
# 2. Dataset path
# ============================================================

DATA_PATH = "./GEFCom2014-L_V2/Load"


tasks = sorted(
    glob.glob(DATA_PATH + "/Task */L*-train.csv")
)

print("Number of tasks:", len(tasks))

Number of tasks: 15


In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_PATH = Path("./GEFCom2014-L_V2/Load")
weather_cols = [f"w{i}" for i in range(1, 26)]


def task_number(path):
    match = re.fullmatch(r"L(\d+)-train", path.stem)
    if match is None:
        raise ValueError(f"Unexpected filename: {path.name}")
    return int(match.group(1))


def parse_task_timestamps(raw, number):
    """
    For the supplied GEFCom2014 archive.

    Raw dates have no separators or zero padding, so individual
    strings can be ambiguous. Reconstruct each task's hourly
    sequence and validate EVERY timestamp against the raw string.
    """
    if number == 1:
        start = pd.Timestamp("2001-01-01 01:00:00")
    else:
        start = (
            pd.Timestamp("2010-10-01 01:00:00")
            + pd.DateOffset(months=number - 2)
        )

    timestamps = pd.date_range(
        start=start,
        periods=len(raw),
        freq="h",
    )

    expected = [
        f"{t.month}{t.day}{t.year} {t.hour}:00"
        for t in timestamps
    ]
    actual = raw.astype(str).str.strip().tolist()

    for row, (observed, required) in enumerate(zip(actual, expected)):
        if observed != required:
            raise ValueError(
                f"Task {number}, row {row}: "
                f"raw timestamp {observed!r} does not match "
                f"expected {required!r}. Check the source file."
            )

    return timestamps


tasks = sorted(
    DATA_PATH.glob("Task */L*-train.csv"),
    key=task_number,
)

if [task_number(p) for p in tasks] != list(range(1, 16)):
    raise ValueError(
        "Expected training files for Tasks 1–15. Check DATA_PATH."
    )

all_data = []

for file in tasks:
    temp = pd.read_csv(file)
    temp["TIMESTAMP"] = parse_task_timestamps(
        temp["TIMESTAMP"],
        task_number(file),
    )
    all_data.append(temp)

df = (
    pd.concat(all_data, ignore_index=True)
    .sort_values(["ZONEID", "TIMESTAMP"])
    .reset_index(drop=True)
)

if df.duplicated(["ZONEID", "TIMESTAMP"]).any():
    raise ValueError("Unexpected duplicate zone/timestamp pairs.")

if df["ZONEID"].nunique() != 1:
    raise ValueError("This pipeline expects one zone.")

# The supplied archive has an initial weather-only period.
# Remove it; no resampling or interpolation is needed.
df_hourly = (
    df.loc[
        df["LOAD"].notna(),
        ["TIMESTAMP", "LOAD"] + weather_cols,
    ]
    .reset_index(drop=True)
)

assert len(df_hourly) == 60600
assert df_hourly[["LOAD"] + weather_cols].notna().all().all()

hourly_steps = df_hourly["TIMESTAMP"].diff().iloc[1:]
assert hourly_steps.eq(pd.Timedelta(hours=1)).all()

print("Raw rows:", len(df))
print("Labelled rows:", len(df_hourly))
print("Start:", df_hourly["TIMESTAMP"].iloc[0])
print("End:", df_hourly["TIMESTAMP"].iloc[-1])
print("Non-hourly gaps:", (~hourly_steps.eq(pd.Timedelta(hours=1))).sum())

Raw rows: 95664
Labelled rows: 60600
Start: 2005-01-01 01:00:00
End: 2011-12-01 00:00:00
Non-hourly gaps: 0


In [3]:
class EnergyDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        df,
        history=168,
        horizon=24
    ):

        self.history = history
        self.horizon = horizon


        # Past demand
        self.demand = (
            df["LOAD_scaled"]
            .values
            .astype(np.float32)
        )


        # Weather
        weather_cols = [
            f"w{i}_scaled"
            for i in range(1,26)
        ]


        self.weather = (
            df[weather_cols]
            .values
            .astype(np.float32)
        )


        # Calendar
        calendar_cols = [
            "hour_sin",
            "hour_cos",
            "dow_sin",
            "dow_cos",
            "month_sin",
            "month_cos"
        ]


        self.calendar = (
            df[calendar_cols]
            .values
            .astype(np.float32)
        )


    def __len__(self):
        return max(
            0,
            len(self.demand) - self.history - self.horizon + 1
        )


    def __getitem__(self, idx):

        start = idx
        end = idx + self.history

        target_start = end
        target_end = end + self.horizon


        # past inputs

        past_demand = self.demand[
            start:end
        ]

        past_weather = self.weather[
            start:end
        ]

        past_calendar = self.calendar[
            start:end
        ]


        # future known variables

        future_weather = self.weather[
            target_start:target_end
        ]

        future_calendar = self.calendar[
            target_start:target_end
        ]


        # previous day load pattern
        # last 24 hours before forecast

        last_day = self.demand[
            end-24:end
        ]


        # target

        target = self.demand[
            target_start:target_end
        ]


        return {

            "past_demand": torch.tensor(
                past_demand,
                dtype=torch.float32
            ).unsqueeze(-1),


            "past_weather": torch.tensor(
                past_weather,
                dtype=torch.float32
            ),


            "past_calendar": torch.tensor(
                past_calendar,
                dtype=torch.float32
            ),


            "future_weather": torch.tensor(
                future_weather,
                dtype=torch.float32
            ),


            "future_calendar": torch.tensor(
                future_calendar,
                dtype=torch.float32
            ),


            "last_day": torch.tensor(
                last_day,
                dtype=torch.float32
            ),


            "target": torch.tensor(
                target,
                dtype=torch.float32
            )

        }

In [4]:
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

HISTORY = 168
HORIZON = 24

df_hourly = (
    df_hourly
    .sort_values("TIMESTAMP")
    .reset_index(drop=True)
    .copy()
)

t = pd.to_datetime(df_hourly["TIMESTAMP"])
df_hourly["TIMESTAMP"] = t

assert t.diff().iloc[1:].eq(pd.Timedelta(hours=1)).all()

# Create calendar features before making split copies.
for name, values, period in [
    ("hour", t.dt.hour, 24),
    ("dow", t.dt.dayofweek, 7),
    ("month", t.dt.month, 12),
]:
    df_hourly[f"{name}_sin"] = np.sin(
        2 * np.pi * values / period
    )
    df_hourly[f"{name}_cos"] = np.cos(
        2 * np.pi * values / period
    )

# 70% training / 7.5% validation / 7.5% calibration / 15% test
n = len(df_hourly)

train_end = int(0.70 * n)
val_end = int(0.775 * n)
cal_end = int(0.85 * n)

split_spec = {
    "n": n,
    "train_end": train_end,
    "val_end": val_end,
    "cal_end": cal_end,
    "first": str(t.iloc[0]),
    "last": str(t.iloc[-1]),
}

# Include history context; targets remain within each block.
train_df = df_hourly.iloc[:train_end].copy()
val_df = df_hourly.iloc[train_end - HISTORY:val_end].copy()
cal_df = df_hourly.iloc[val_end - HISTORY:cal_end].copy()
test_df = df_hourly.iloc[cal_end - HISTORY:].copy()

weather_cols = [f"w{i}" for i in range(1, 26)]
scaled_weather_cols = [f"{c}_scaled" for c in weather_cols]

load_scaler = StandardScaler().fit(train_df[["LOAD"]])
weather_scaler = StandardScaler().fit(train_df[weather_cols])

for frame in [train_df, val_df, cal_df, test_df]:
    frame["LOAD_scaled"] = load_scaler.transform(
        frame[["LOAD"]]
    ).ravel()

    frame[scaled_weather_cols] = weather_scaler.transform(
        frame[weather_cols]
    )

    assert np.isfinite(
        frame[["LOAD_scaled"] + scaled_weather_cols]
    ).all().all()

train_dataset = EnergyDataset(
    train_df, history=HISTORY, horizon=HORIZON
)
val_dataset = EnergyDataset(
    val_df, history=HISTORY, horizon=HORIZON
)
cal_dataset = EnergyDataset(
    cal_df, history=HISTORY, horizon=HORIZON
)
test_dataset = EnergyDataset(
    test_df, history=HISTORY, horizon=HORIZON
)

for name, frame, dataset in [
    ("Train", train_df, train_dataset),
    ("Validation", val_df, val_dataset),
    ("Calibration", cal_df, cal_dataset),
    ("Test", test_df, test_dataset),
]:
    assert len(dataset) == len(frame) - HISTORY - HORIZON + 1

    print(
        name,
        "| Windows:", len(dataset),
        "| First target:", frame["TIMESTAMP"].iloc[HISTORY],
    )

train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=64, shuffle=False, num_workers=0
)
cal_loader = DataLoader(
    cal_dataset, batch_size=64, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=64, shuffle=False, num_workers=0
)

Train | Windows: 42229 | First target: 2005-01-08 01:00:00
Validation | Windows: 4522 | First target: 2009-11-03 13:00:00
Calibration | Windows: 4522 | First target: 2010-05-11 22:00:00
Test | Windows: 9067 | First target: 2010-11-17 07:00:00


In [5]:
class DomainEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=32
    ):
        super().__init__()


        self.projection = nn.Linear(
            input_dim,
            d_model
        )


        self.norm = nn.LayerNorm(
            d_model
        )


    def forward(self,x):

        # x:
        # batch, time, features

        x = self.projection(x)

        x = self.norm(x)

        return x

In [6]:
class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=500
    ):

        super().__init__()


        pe = torch.zeros(
            max_len,
            d_model
        )


        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)


        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            *
            (-np.log(10000.0)/d_model)
        )


        pe[:,0::2] = torch.sin(
            position*div_term
        )


        pe[:,1::2] = torch.cos(
            position*div_term
        )


        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )


    def forward(self,x):

        return x + self.pe[:,:x.size(1)]

In [7]:
class DomainFusion(nn.Module):

    def __init__(
        self,
        d_model=64,
        heads=4
    ):

        super().__init__()


        # weather and calendar gates

        self.weather_gate = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )


        self.calendar_gate = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )


        self.attention = nn.MultiheadAttention(
            d_model,
            heads,
            batch_first=True
        )


        self.norm = nn.LayerNorm(
            d_model
        )


        self.dropout = nn.Dropout(0.1)



    def forward(
        self,
        demand,
        weather,
        calendar
    ):


        w_gate = self.weather_gate(
            weather
        )


        c_gate = self.calendar_gate(
            calendar
        )


        # demand is preserved

        x = (
            demand
            +
            w_gate * weather
            +
            c_gate * calendar
        )


        attn,_ = self.attention(
            x,
            x,
            x
        )


        x = self.norm(
            x+self.dropout(attn)
        )


        return x

In [8]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


class OrderedQuantileHead(nn.Module):
    """
    Input:  [batch, horizon, d_model]
    Output: [batch, horizon, 99]

    Quantiles: 0.01, 0.02, ..., 0.99
    Median index: 49

    Positive increments enforce non-crossing quantiles.
    """

    def __init__(self, input_dim):
        super().__init__()

        self.hidden = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.GELU(),
        )
        self.output = nn.Linear(128, 99)

        # Initially small corrections to the previous-day baseline.
        nn.init.normal_(self.output.weight, std=0.005)

        with torch.no_grad():
            self.output.bias.zero_()

            # Initial adjacent quantile spacing in standardized units.
            initial_spacing = 0.02
            inverse_softplus = math.log(math.expm1(initial_spacing))
            self.output.bias[1:].fill_(inverse_softplus)

    def forward(self, x):
        raw = self.output(self.hidden(x))

        median = raw[..., :1]

        # 49 increments on either side of the median.
        lower_steps = F.softplus(raw[..., 1:50])
        upper_steps = F.softplus(raw[..., 50:99])

        lower = median - torch.cumsum(
            lower_steps, dim=-1
        ).flip(-1)

        upper = median + torch.cumsum(
            upper_steps, dim=-1
        )

        return torch.cat(
            [lower, median, upper],
            dim=-1,
        )


class DreamFormerV5(nn.Module):

    def __init__(self, d_model=64, horizon=24):
        super().__init__()

        if horizon != 24:
            raise ValueError(
                "The current last-day dataset requires horizon=24."
            )

        self.register_buffer(
            "taus",
            torch.arange(1, 100).float() / 100,
        )

        self.demand_encoder = DomainEncoder(1, d_model)
        self.weather_encoder = DomainEncoder(25, d_model)
        self.calendar_encoder = DomainEncoder(6, d_model)

        self.future_weather_encoder = DomainEncoder(25, d_model)
        self.future_calendar_encoder = DomainEncoder(6, d_model)

        self.position = PositionalEncoding(d_model)
        self.fusion = DomainFusion(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=256,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
        )

        self.query = nn.Parameter(
            torch.randn(horizon, d_model)
        )

        self.future_fusion = nn.Linear(
            2 * d_model,
            d_model,
        )
        self.future_norm = nn.LayerNorm(d_model)

        self.head = OrderedQuantileHead(d_model)

    def forward(
        self,
        past_demand,
        past_weather,
        past_calendar,
        future_weather,
        future_calendar,
        last_day,
    ):
        # Historical encoders and fusion: same structure as v4.
        d = self.position(self.demand_encoder(past_demand))
        w = self.position(self.weather_encoder(past_weather))
        c = self.position(self.calendar_encoder(past_calendar))

        history = self.transformer(
            self.fusion(d, w, c)
        )

        # Horizon-specific retrieval: same structure as v4.
        queries = self.query.unsqueeze(0).expand(
            history.size(0), -1, -1
        )

        attention = torch.softmax(
            queries @ history.transpose(1, 2)
            / math.sqrt(history.size(-1)),
            dim=-1,
        )

        context = attention @ history

        # Future weather and calendar inputs.
        future = self.future_fusion(
            torch.cat(
                [
                    self.future_weather_encoder(future_weather),
                    self.future_calendar_encoder(future_calendar),
                ],
                dim=-1,
            )
        )

        z = self.future_norm(context + future)

        # Add the same previous-day baseline to every quantile.
        # Output: [batch, 24, 99]
        return last_day.unsqueeze(-1) + self.head(z)


def pinball_loss(quantiles, target, taus):
    """
    quantiles: [batch, horizon, 99]
    target:    [batch, horizon]
    taus:      [99]
    """
    error = target.unsqueeze(-1) - quantiles

    return torch.maximum(
        taus * error,
        (taus - 1) * error,
    ).mean()


INPUT_KEYS = (
    "past_demand",
    "past_weather",
    "past_calendar",
    "future_weather",
    "future_calendar",
    "last_day",
)


def batch_inputs(batch, device):
    return [
        batch[key].to(device)
        for key in INPUT_KEYS
    ]

In [9]:
import random
import time

from pathlib import Path
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

config = {
    "d_model": 64,
    "horizon": 24,
}

model = DreamFormerV5(**config).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
)

CHECKPOINT = Path(
    f"dreamformer_v51_raw_seed{SEED}_best.pt"
).resolve()

CALIBRATED_CHECKPOINT = Path(
    f"dreamformer_v51_calibrated_seed{SEED}.pt"
).resolve()

EPOCHS = 30
PATIENCE = 8

print("Raw checkpoint:", CHECKPOINT)


def run_epoch(loader, training):
    model.train(training)

    total = 0.0
    count = 0

    progress = tqdm(
        loader,
        desc="Train" if training else "Validation",
        leave=False,
    )

    with torch.set_grad_enabled(training):
        for batch in progress:
            target = batch["target"].to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            q = model(*batch_inputs(batch, device))
            loss = pinball_loss(q, target, model.taus)

            if not torch.isfinite(loss).item():
                raise FloatingPointError("Non-finite loss")

            if training:
                loss.backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0
                )

                optimizer.step()

            size = len(target)
            total += loss.detach().item() * size
            count += size

            progress.set_postfix(
                pinball=f"{total / count:.5f}"
            )

    if count == 0:
        raise ValueError("Empty loader")

    return total / count


best_val = float("inf")
bad_epochs = 0

history = {
    "train_pinball": [],
    "val_pinball": [],
}

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_score = run_epoch(train_loader, True)
    val_score = run_epoch(val_loader, False)

    scheduler.step(val_score)

    history["train_pinball"].append(train_score)
    history["val_pinball"].append(val_score)

    improved = val_score < best_val

    if improved:
        best_val = val_score
        bad_epochs = 0

        state = {
            "version": "v5.1-raw",
            "epoch": epoch,
            "config": config,
            "seed": SEED,
            "split_spec": split_spec,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "val_pinball": val_score,
            "load_mean": load_scaler.mean_.tolist(),
            "load_scale": load_scaler.scale_.tolist(),
            "weather_mean": weather_scaler.mean_.tolist(),
            "weather_scale": weather_scaler.scale_.tolist(),
        }

        # Replace the previous checkpoint only after writing succeeds.
        temporary = CHECKPOINT.with_suffix(".tmp")
        torch.save(state, temporary)
        temporary.replace(CHECKPOINT)

    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train: {train_score:.6f} | "
        f"Val: {val_score:.6f} | "
        f"Time: {time.time() - start:.1f}s"
        + (" | SAVED" if improved else "")
    )

    if bad_epochs >= PATIENCE:
        print("Early stopping.")
        break

/opt/anaconda3/envs/AI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw checkpoint: /Users/pratyush/Coding/Projects/dreamformer/dreamformer_v51_raw_seed42_best.pt


Epoch 1/30 | Train: 0.107630 | Val: 0.110565 | Time: 75.0s | SAVED


Epoch 2/30 | Train: 0.079245 | Val: 0.088066 | Time: 106.3s | SAVED


Epoch 3/30 | Train: 0.067898 | Val: 0.079536 | Time: 93.9s | SAVED


Epoch 4/30 | Train: 0.060801 | Val: 0.071589 | Time: 99.1s | SAVED


... (21 outputs left out)

Epoch 26/30 | Train: 0.035721 | Val: 0.053179 | Time: 78.0s


Epoch 27/30 | Train: 0.035493 | Val: 0.053075 | Time: 91.5s


Epoch 28/30 | Train: 0.035322 | Val: 0.052295 | Time: 116.0s


Epoch 29/30 | Train: 0.035131 | Val: 0.052950 | Time: 135.0s
Early stopping.


In [10]:
from pathlib import Path
import torch

expected_path = Path(CHECKPOINT).resolve()

print("Current CHECKPOINT:", globals().get("CHECKPOINT"))
print("Expected checkpoint:", expected_path)
print("File exists:", expected_path.is_file())

if not expected_path.is_file():
    raise FileNotFoundError(expected_path)

saved = torch.load(
    expected_path,
    map_location="cpu",
    weights_only=True,
)

print("Saved object type:", type(saved).__name__)

if not isinstance(saved, dict):
    raise TypeError("Expected a checkpoint dictionary.")

print("Available keys:", list(saved.keys()))
print("Version:", saved.get("version", "<missing>"))
print("Epoch:", saved.get("epoch", "<missing>"))
print("Validation pinball:", saved.get("val_pinball", "<missing>"))
print("Split specification:", saved.get("split_spec", "<missing>"))

required = {
    "version",
    "split_spec",
    "model_state_dict",
    "load_mean",
    "load_scale",
    "weather_mean",
    "weather_scale",
}

missing = required.difference(saved)

if missing:
    print("\nMissing required fields:", sorted(missing))
    print("Keep this file unchanged until its origin is identified.")
else:
    assert saved["version"] == "v5.1-raw"
    assert saved["split_spec"] == split_spec

    CHECKPOINT = expected_path
    print("\nCheckpoint metadata matches. Rerun the calibration cell.")

Current CHECKPOINT: /Users/pratyush/Coding/Projects/dreamformer/dreamformer_v51_raw_seed42_best.pt
Expected checkpoint: /Users/pratyush/Coding/Projects/dreamformer/dreamformer_v51_raw_seed42_best.pt
File exists: True
Saved object type: dict
Available keys: ['version', 'epoch', 'config', 'seed', 'split_spec', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'val_pinball', 'load_mean', 'load_scale', 'weather_mean', 'weather_scale']
Version: v5.1-raw
Epoch: 21
Validation pinball: 0.051843991237027306
Split specification: {'n': 60600, 'train_end': 42420, 'val_end': 46965, 'cal_end': 51510, 'first': '2005-01-01 01:00:00', 'last': '2011-12-01 00:00:00'}

Checkpoint metadata matches. Rerun the calibration cell.


In [11]:
checkpoint = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=True,
)

assert checkpoint["version"] == "v5.1-raw"
assert checkpoint["split_spec"] == split_spec

for scaler, prefix in [
    (load_scaler, "load"),
    (weather_scaler, "weather"),
]:
    assert np.allclose(
        scaler.mean_,
        checkpoint[f"{prefix}_mean"],
    )
    assert np.allclose(
        scaler.scale_,
        checkpoint[f"{prefix}_scale"],
    )

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)
model.eval()

print("Loaded epoch:", checkpoint["epoch"])


@torch.no_grad()
def predict_raw(loader):
    model.eval()

    predictions = []
    targets = []

    for batch in loader:
        q = model(*batch_inputs(batch, device))

        predictions.append(q.cpu().numpy())
        targets.append(batch["target"].cpu().numpy())

    q = np.concatenate(predictions, axis=0)
    y = np.concatenate(targets, axis=0)

    scale = float(load_scaler.scale_[0])
    mean = float(load_scaler.mean_[0])

    return q * scale + mean, y * scale + mean


def fit_side_scale(offsets, residual, levels):
    """
    Minimize average pinball loss over a single positive scale.

    offsets: [samples, 49], quantile minus median
    residual: [samples], actual minus median
    levels: [49], corresponding quantile levels

    The objective is convex and piecewise linear.
    Its minimizer is a weighted quantile of residual/offset.
    """
    d = np.asarray(offsets, dtype=np.float64)

    r = np.broadcast_to(
        np.asarray(residual, dtype=np.float64)[:, None],
        d.shape,
    )

    tau = np.broadcast_to(
        np.asarray(levels, dtype=np.float64),
        d.shape,
    )

    mask = d != 0

    if not mask.any():
        return 1.0

    weight = np.abs(d[mask])
    ratio = r[mask] / d[mask]

    alpha = np.where(
        d[mask] > 0,
        tau[mask],
        1 - tau[mask],
    )

    order = np.argsort(ratio)
    cumulative = np.cumsum(weight[order])
    target_weight = np.sum(weight * alpha)

    index = min(
        np.searchsorted(cumulative, target_weight),
        len(order) - 1,
    )

    return max(float(ratio[order[index]]), 1e-6)


def apply_calibration(q, lower_scale, upper_scale):
    result = q.copy()
    median = q[..., 49:50]

    result[..., :49] = (
        median
        + lower_scale[None, :, None]
        * (q[..., :49] - median)
    )

    result[..., 50:] = (
        median
        + upper_scale[None, :, None]
        * (q[..., 50:] - median)
    )

    # Median index 49 is never modified.
    assert np.isfinite(result).all()
    assert np.array_equal(result[..., 49], q[..., 49])
    assert (np.diff(result, axis=-1) >= -1e-5).all()

    return result


# Calibration data only: no test predictions or targets used here.
cal_q, cal_y = predict_raw(cal_loader)

assert np.isfinite(cal_q).all()
assert np.isfinite(cal_y).all()

taus = np.arange(1, 100) / 100.0

lower_scale = np.ones(HORIZON)
upper_scale = np.ones(HORIZON)

for h in range(HORIZON):
    median = cal_q[:, h, 49].astype(np.float64)
    residual = cal_y[:, h].astype(np.float64) - median

    lower_scale[h] = fit_side_scale(
        cal_q[:, h, :49].astype(np.float64) - median[:, None],
        residual,
        taus[:49],
    )

    upper_scale[h] = fit_side_scale(
        cal_q[:, h, 50:].astype(np.float64) - median[:, None],
        residual,
        taus[50:],
    )

cal_q_adjusted = apply_calibration(
    cal_q,
    lower_scale,
    upper_scale,
)

# Save model weights, preprocessing metadata, and calibration together.
bundle = dict(checkpoint)
bundle["version"] = "v5.1-calibrated"
bundle["lower_scale"] = lower_scale.tolist()
bundle["upper_scale"] = upper_scale.tolist()

temporary = CALIBRATED_CHECKPOINT.with_suffix(".tmp")
torch.save(bundle, temporary)
temporary.replace(CALIBRATED_CHECKPOINT)

print("Saved calibrated checkpoint:", CALIBRATED_CHECKPOINT)

print(
    pd.DataFrame({
        "horizon": np.arange(1, HORIZON + 1),
        "lower_scale": lower_scale,
        "upper_scale": upper_scale,
    })
    .round(3)
    .to_string(index=False)
)

Loaded epoch: 21
Saved calibrated checkpoint: /Users/pratyush/Coding/Projects/dreamformer/dreamformer_v51_calibrated_seed42.pt
 horizon  lower_scale  upper_scale
       1        1.338        0.882
       2        1.374        0.843
       3        1.445        0.820
       4        1.492        0.813
       5        1.470        0.828
       6        1.541        0.839
       7        1.521        0.882
       8        1.482        0.915
       9        1.514        0.915
      10        1.490        0.981
      11        1.475        0.980
      12        1.425        1.017
      13        1.443        1.005
      14        1.444        1.060
      15        1.423        1.077
      16        1.378        1.084
      17        1.325        1.105
      18        1.314        1.075
      19        1.301        1.074
      20        1.287        1.069
      21        1.315        1.033
      22        1.279        1.003
      23        1.332        0.972
      24        1.314        0.99

In [12]:
bundle = torch.load(
    CALIBRATED_CHECKPOINT,
    map_location=device,
    weights_only=True,
)

assert bundle["version"] == "v5.1-calibrated"
assert bundle["split_spec"] == split_spec

model.load_state_dict(
    bundle["model_state_dict"],
    strict=True,
)
model.eval()

lower_scale = np.asarray(bundle["lower_scale"])
upper_scale = np.asarray(bundle["upper_scale"])

raw_quantile_pred, y_true = predict_raw(test_loader)

quantile_pred = apply_calibration(
    raw_quantile_pred,
    lower_scale,
    upper_scale,
)

# Preserve names expected by your existing v5 plotting cell.
pred = quantile_pred[..., 49]
taus = np.arange(1, 100) / 100.0


def mean_pinball_numpy(q, y):
    total = 0.0
    levels = np.arange(1, 100) / 100.0

    # Compute in chunks to limit memory use.
    for start in range(0, len(y), 256):
        error = (
            y[start:start + 256, :, None]
            - q[start:start + 256]
        )

        total += np.maximum(
            levels * error,
            (levels - 1) * error,
        ).sum(dtype=np.float64)

    return total / q.size


def report(q, y, label):
    error = q[..., 49] - y
    valid = np.abs(y) > 1e-8

    mape = (
        100 * np.abs(error[valid] / y[valid]).mean()
        if valid.any()
        else float("nan")
    )

    print("\n" + "=" * 65)
    print(label)
    print(f"Median MAE: {np.abs(error).mean():.4f}")
    print(f"Median RMSE: {np.sqrt((error ** 2).mean()):.4f}")
    print(f"Median MAPE: {mape:.4f}%")
    print(f"Mean pinball: {mean_pinball_numpy(q, y):.4f}")

    rows = []

    for lo, hi in [(10, 90), (5, 95), (1, 99)]:
        lower = q[..., lo - 1]
        upper = q[..., hi - 1]

        rows.append({
            "interval": f"{hi - lo}%",
            "coverage_pct": 100 * (
                (y >= lower) & (y <= upper)
            ).mean(),
            "below_lower_pct": 100 * (y < lower).mean(),
            "above_upper_pct": 100 * (y > upper).mean(),
            "mean_width": (upper - lower).mean(),
        })

    print(
        pd.DataFrame(rows)
        .round(3)
        .to_string(index=False)
    )


report(
    raw_quantile_pred,
    y_true,
    "TEST: RAW v5.1",
)

report(
    quantile_pred,
    y_true,
    "TEST: CALIBRATED v5.1",
)

assert np.array_equal(
    pred,
    raw_quantile_pred[..., 49],
)

print("\nMedian forecasts are exactly unchanged by calibration.")


TEST: RAW v5.1
Median MAE: 7.3850
Median RMSE: 11.5529
Median MAPE: 6.0748%
Mean pinball: 2.7416
interval  coverage_pct  below_lower_pct  above_upper_pct  mean_width
     80%        63.414           27.305            9.281   15.674000
     90%        76.037           18.122            5.841   20.431999
     98%        91.521            6.595            1.884   30.888000

TEST: CALIBRATED v5.1
Median MAE: 7.3850
Median RMSE: 11.5529
Median MAPE: 6.0748%
Mean pinball: 2.6360
interval  coverage_pct  below_lower_pct  above_upper_pct  mean_width
     80%        74.528           15.748            9.724   18.594000
     90%        85.739            8.022            6.238   24.228001
     98%        95.894            2.001            2.105   36.471001

Median forecasts are exactly unchanged by calibration.


In [13]:
report(cal_q, cal_y, "CALIBRATION BLOCK: RAW")
report(cal_q_adjusted, cal_y, "CALIBRATION BLOCK: ADJUSTED")



CALIBRATION BLOCK: RAW
Median MAE: 5.5658
Median RMSE: 7.6900
Median MAPE: 3.4538%
Mean pinball: 2.0125
interval  coverage_pct  below_lower_pct  above_upper_pct  mean_width
     80%        72.650           17.118           10.231   14.632000
     90%        83.835           10.444            5.721   19.077999
     98%        95.233            3.240            1.527   28.905001

CALIBRATION BLOCK: ADJUSTED
Median MAE: 5.5658
Median RMSE: 7.6900
Median MAPE: 3.4538%
Mean pinball: 1.9803
interval  coverage_pct  below_lower_pct  above_upper_pct  mean_width
     80%        80.510            8.651           10.839   17.441999
     90%        89.756            4.097            6.148   22.725000
     98%        97.403            0.881            1.717   34.320999
